# Notebook 02 — Temporal Decomposition

**Repo:** `temporal-phase-lock`  
**Notebook:** `notebooks/02_temporal_decomposition.ipynb`  
**Purpose:** turn adaptive temporal reasoning into measurable anchor/constraint structure.

Notebook 01 created a transparent AdapTime-style baseline:

```text
question → reformulate → rewrite → review
```

Notebook 02 expands that into decomposition measurement:

```text
question
  → detect temporal operators
  → extract anchors
  → build decomposition graph
  → score decomposition completeness
  → flag missing or ambiguous constraints
```

Core idea:

> AdapTime says: decompose and review.  
> Temporal Phase Lock says: measure decomposition before trusting review.

## 0. Environment setup

In [ ]:
import os, re, math, json, random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

REPO_NAME = "temporal-phase-lock"
NOTEBOOK_ID = "02_temporal_decomposition"

BASE_DIR = Path(".")
FIG_DIR = BASE_DIR / "figures"
DOCS_DIR = BASE_DIR / "docs"
ARTIFACT_DIR = BASE_DIR / "artifacts"

for d in [FIG_DIR, DOCS_DIR, ARTIFACT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Ready: {REPO_NAME}/{NOTEBOOK_ID}")

## 1. Recreate Notebook 01 temporal QA cases

In [ ]:
@dataclass
class TemporalCase:
    case_id: str
    question: str
    evidence: List[str]
    gold: str
    answer_type: str

CASES = [
    TemporalCase("TQA-001","Who joined after Mira but before Theo?",
        ["Mira joined the project in March.","Jon joined the project in May.","Theo joined the project in August.","Leah joined the project in November."],
        "Jon","between"),
    TemporalCase("TQA-002","Which event happened first?",
        ["The calibration finished on Tuesday.","The sensor test started on Monday.","The report was drafted on Wednesday."],
        "sensor test","first"),
    TemporalCase("TQA-003","What happened immediately after the review?",
        ["The draft happened before the review.","The review happened before the revision.","The revision happened before the release."],
        "revision","after"),
    TemporalCase("TQA-004","Which item was completed last?",
        ["Notebook 01 was completed before Notebook 02.","Notebook 03 was completed after Notebook 02.","Notebook 04 was completed after Notebook 03."],
        "Notebook 04","last"),
    TemporalCase("TQA-005","Did the audit occur before deployment?",
        ["The prototype was tested in April.","The audit occurred in June.","Deployment occurred in September."],
        "Yes","yes_no"),
]
case_df = pd.DataFrame([asdict(c) for c in CASES])
case_df

## 2. Temporal operator inventory

In [ ]:
TEMPORAL_OPERATORS = ["immediately after", "after", "before", "between", "first", "last"]

def detect_operators(question: str) -> List[str]:
    q = question.lower()
    ops = []
    if q.startswith("did") and "before" in q: ops.append("yes_no_before")
    if "after" in q and "before" in q: ops.append("between")
    if "immediately after" in q: ops.append("immediately_after")
    if "happened first" in q or "event happened first" in q or " first" in q: ops.append("first")
    if "completed last" in q or " last" in q: ops.append("last")
    for op in TEMPORAL_OPERATORS:
        tag = op.replace(" ", "_")
        if op in q and tag not in ops:
            ops.append(tag)
    clean = []
    for op in ops:
        if op not in clean:
            clean.append(op)
    return clean

operator_df = pd.DataFrame([{
    "case_id": c.case_id,
    "question": c.question,
    "operators": ", ".join(detect_operators(c.question)),
    "answer_type": c.answer_type,
} for c in CASES])
operator_df

## 3. Anchor extraction from questions

In [ ]:
def clean_phrase(text: str) -> str:
    text = re.sub(r"[\?\.]$", "", text.strip())
    text = re.sub(r"^(the|a|an)\s+", "", text, flags=re.I)
    return text

def extract_question_anchors(question: str) -> Dict:
    q = question.strip()
    ql = q.lower()
    m = re.search(r"after\s+([A-Za-z0-9 ]+?)\s+but\s+before\s+([A-Za-z0-9 ]+)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1)), clean_phrase(m.group(2))],
                "target":"entity_between_anchors","relation":"after_left_and_before_right","required_anchor_count":2}
    m = re.search(r"immediately after\s+(.+?)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1))],
                "target":"successor_event","relation":"immediate_successor","required_anchor_count":1}
    m = re.match(r"did\s+(.+?)\s+occur\s+before\s+(.+?)\?", q, flags=re.I)
    if m:
        return {"anchors":[clean_phrase(m.group(1)), clean_phrase(m.group(2))],
                "target":"yes_no_comparison","relation":"before","required_anchor_count":2}
    if "first" in ql:
        return {"anchors":["timeline_start"],"target":"earliest_event","relation":"min_time","required_anchor_count":1}
    if "last" in ql:
        return {"anchors":["timeline_end"],"target":"latest_event","relation":"max_time","required_anchor_count":1}
    return {"anchors":[],"target":"unknown","relation":"unknown","required_anchor_count":0}

anchor_df = pd.DataFrame([{
    "case_id": c.case_id,
    "question": c.question,
    "anchors": ", ".join(extract_question_anchors(c.question)["anchors"]),
    "target": extract_question_anchors(c.question)["target"],
    "relation": extract_question_anchors(c.question)["relation"],
    "required_anchor_count": extract_question_anchors(c.question)["required_anchor_count"],
} for c in CASES])
anchor_df

## 4. Evidence decomposition

In [ ]:
MONTH_ORDER = {"january":1,"february":2,"march":3,"april":4,"may":5,"june":6,"july":7,"august":8,"september":9,"october":10,"november":11,"december":12}
WEEKDAY_ORDER = {"monday":1,"tuesday":2,"wednesday":3,"thursday":4,"friday":5,"saturday":6,"sunday":7}

def extract_subject(sentence: str) -> str:
    parts = re.split(r"\s+(joined|occurred|finished|started|was|were|completed|happened|drafted|tested)\b",
                     sentence, maxsplit=1, flags=re.I)
    return clean_phrase(parts[0])

def extract_time_fact(sentence: str):
    s = sentence.lower()
    for month, idx in MONTH_ORDER.items():
        if month in s:
            return extract_subject(sentence), idx, month.title()
    for day, idx in WEEKDAY_ORDER.items():
        if day in s:
            return extract_subject(sentence), idx, day.title()
    return None

def extract_directed_edges(sentence: str):
    s = sentence.strip().rstrip(".")
    patterns = [
        (r"(.+?)\s+was completed before\s+(.+)", False),
        (r"(.+?)\s+was completed after\s+(.+)", True),
        (r"(.+?)\s+happened before\s+(.+)", False),
        (r"(.+?)\s+happened after\s+(.+)", True),
    ]
    for pat, reverse in patterns:
        m = re.match(pat, s, flags=re.I)
        if m:
            a, b = clean_phrase(m.group(1)), clean_phrase(m.group(2))
            return [(b, "before", a)] if reverse else [(a, "before", b)]
    return []

def topological_order_from_edges(edges):
    nodes = sorted(set([a for a,_,b in edges] + [b for a,_,b in edges]))
    incoming = {n:set() for n in nodes}
    outgoing = {n:set() for n in nodes}
    for a, rel, b in edges:
        if rel == "before":
            outgoing[a].add(b); incoming[b].add(a)
    order, ready = [], sorted([n for n in nodes if not incoming[n]])
    while ready:
        n = ready.pop(0); order.append(n)
        for m in sorted(outgoing[n]):
            incoming[m].discard(n)
            if not incoming[m]: ready.append(m)
        ready = sorted(set(ready))
    return order

def decompose_evidence(case: TemporalCase) -> Dict:
    facts, fact_labels, explicit_edges, rows = {}, {}, [], []
    for sent in case.evidence:
        tf = extract_time_fact(sent)
        if tf:
            subject, idx, label = tf
            facts[subject] = idx
            fact_labels[subject] = label
            rows.append({"case_id":case.case_id,"source":subject,"relation":"has_time","target":label,
                         "evidence_sentence":sent,"edge_type":"fact"})
        for a, rel, b in extract_directed_edges(sent):
            explicit_edges.append((a, rel, b))
            rows.append({"case_id":case.case_id,"source":a,"relation":rel,"target":b,
                         "evidence_sentence":sent,"edge_type":"constraint"})
    derived_edges = []
    if facts:
        timeline = [k for k,v in sorted(facts.items(), key=lambda kv: kv[1])]
        for a, b in zip(timeline[:-1], timeline[1:]):
            derived_edges.append((a, "before", b))
            rows.append({"case_id":case.case_id,"source":a,"relation":"before","target":b,
                         "evidence_sentence":"derived from ordered time facts","edge_type":"derived"})
    else:
        timeline = topological_order_from_edges(explicit_edges) if explicit_edges else []
    return {"facts":facts, "fact_labels":fact_labels, "explicit_edges":explicit_edges,
            "derived_edges":derived_edges, "all_edges":explicit_edges+derived_edges,
            "timeline":timeline, "rows":rows}

all_edge_rows = []
for c in CASES:
    all_edge_rows.extend(decompose_evidence(c)["rows"])
edge_df = pd.DataFrame(all_edge_rows)
edge_df

## 5. Decomposition graph table

In [ ]:
graph_table = edge_df[["case_id","source","relation","target","edge_type","evidence_sentence"]].copy()
graph_table

## 6. Completeness scoring

In [ ]:
def loose_contains(name: str, collection: List[str]) -> bool:
    n = name.lower().strip()
    return any(n in item.lower() or item.lower() in n for item in collection)

def score_case_decomposition(case: TemporalCase) -> Dict:
    qa = extract_question_anchors(case.question)
    ev = decompose_evidence(case)
    timeline, edges = ev["timeline"], ev["all_edges"]
    required = qa["anchors"]
    required_n = qa["required_anchor_count"]
    if required in (["timeline_start"], ["timeline_end"]):
        recovered_n = 1 if len(timeline) >= 1 else 0
    else:
        recovered_n = sum(1 for a in required if loose_contains(a, timeline))
    anchor_score = recovered_n / required_n if required_n else 1.0
    min_edges = 2 if case.answer_type == "between" else 1
    edge_n = len(edges)
    constraint_coverage = min(1.0, edge_n / min_edges) if min_edges else 1.0
    combined = 0.5 * anchor_score + 0.5 * constraint_coverage
    flags = []
    if required_n and recovered_n < required_n: flags.append("missing_anchor")
    if edge_n < min_edges: flags.append("insufficient_edges")
    if not timeline: flags.append("empty_timeline")
    return {
        "case_id": case.case_id,
        "answer_type": case.answer_type,
        "required_anchors": ", ".join(required),
        "recovered_anchor_count": recovered_n,
        "required_anchor_count": required_n,
        "anchor_score": anchor_score,
        "recovered_edges": edge_n,
        "minimum_required_edges": min_edges,
        "constraint_coverage": constraint_coverage,
        "temporal_decomposition_score": combined,
        "ambiguity_flags": ", ".join(flags) if flags else "none",
        "timeline": " → ".join(timeline),
    }

score_df = pd.DataFrame([score_case_decomposition(c) for c in CASES])
score_df

## 7. Figure 1 — decomposition score by case

In [ ]:
plt.figure(figsize=(8, 4.8))
plt.bar(score_df["case_id"], score_df["temporal_decomposition_score"])
plt.ylim(0, 1.05)
plt.title("Temporal decomposition score by case")
plt.xlabel("Case")
plt.ylabel("Score")
plt.xticks(rotation=20)
plt.tight_layout()
fig1_path = FIG_DIR / "02_decomposition_scores.png"
plt.savefig(fig1_path, dpi=180, bbox_inches="tight")
plt.show()
fig1_path

## 8. Figure 2 — anchor / edge recovery matrix

In [ ]:
matrix_cols = ["anchor_score", "constraint_coverage", "temporal_decomposition_score"]
matrix = score_df[matrix_cols].to_numpy()
plt.figure(figsize=(8, 4.8))
plt.imshow(matrix, aspect="auto", vmin=0, vmax=1)
plt.colorbar(label="score")
plt.yticks(np.arange(len(score_df)), score_df["case_id"])
plt.xticks(np.arange(len(matrix_cols)), ["anchor", "edge", "combined"], rotation=20)
plt.title("Anchor / edge recovery matrix")
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        plt.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center")
plt.tight_layout()
fig2_path = FIG_DIR / "02_anchor_edge_matrix.png"
plt.savefig(fig2_path, dpi=180, bbox_inches="tight")
plt.show()
fig2_path

## 9. Figure 3 — example temporal decomposition graph

In [ ]:
example = decompose_evidence(CASES[0])
nodes = example["timeline"]
plt.figure(figsize=(9.5, 2.8))
x = np.arange(len(nodes)); y = np.zeros(len(nodes))
plt.plot(x, y, marker="o", linewidth=2)
for i, node in enumerate(nodes):
    plt.text(i, 0.08, node, ha="center", va="bottom", fontsize=12)
    if i < len(nodes) - 1:
        plt.text(i + 0.5, -0.08, "before", ha="center", va="top", fontsize=10)
        plt.annotate("", xy=(i + 0.85, 0), xytext=(i + 0.15, 0),
                     arrowprops=dict(arrowstyle="->", lw=1.5))
plt.ylim(-0.25, 0.35)
plt.yticks([]); plt.xticks([])
plt.title("Example temporal decomposition graph: TQA-001")
plt.tight_layout()
fig3_path = FIG_DIR / "02_example_decomposition_graph.png"
plt.savefig(fig3_path, dpi=180, bbox_inches="tight")
plt.show()
fig3_path

## 10. Save artifacts

In [ ]:
cases_path = ARTIFACT_DIR / "02_temporal_decomposition_cases.csv"
edges_path = ARTIFACT_DIR / "02_temporal_decomposition_edges.csv"
scores_path = ARTIFACT_DIR / "02_temporal_decomposition_scores.csv"
summary_path = DOCS_DIR / "02_temporal_decomposition_summary.md"

case_df.to_csv(cases_path, index=False)
graph_table.to_csv(edges_path, index=False)
score_df.to_csv(scores_path, index=False)

mean_score = score_df["temporal_decomposition_score"].mean()
min_score = score_df["temporal_decomposition_score"].min()
num_flags = int((score_df["ambiguity_flags"] != "none").sum())

summary_md = f"""# Notebook 02 — Temporal Decomposition

Repo: `{REPO_NAME}`  
Notebook: `notebooks/02_temporal_decomposition.ipynb`

## Purpose

This notebook turns adaptive temporal reasoning into measurable anchor/constraint structure.

## Summary metrics

| Metric | Value |
|---|---:|
| Mean temporal decomposition score | {mean_score:.3f} |
| Minimum temporal decomposition score | {min_score:.3f} |
| Cases with ambiguity flags | {num_flags} |
| Number of cases | {len(score_df)} |
| Number of graph rows | {len(graph_table)} |

## Figures

- `figures/02_decomposition_scores.png`
- `figures/02_anchor_edge_matrix.png`
- `figures/02_example_decomposition_graph.png`

## Handoff to Notebook 03

Notebook 03 should introduce phase-lock validation using:

```text
cos(theta) >= 1 / sqrt(1^2 + 1^2)
```
"""
summary_path.write_text(summary_md)

print(cases_path)
print(edges_path)
print(scores_path)
print(summary_path)

## 11. Handoff to Notebook 03

In [ ]:
PHASE_LOCK_GATE = 1 / math.sqrt(1**2 + 1**2)

handoff_features = score_df.copy()
handoff_features["operator_count"] = operator_df["operators"].apply(lambda s: len([x for x in s.split(", ") if x]))
handoff_features["timeline_length"] = handoff_features["timeline"].apply(lambda s: 0 if not s else len(s.split(" → ")))

feature_cols = ["anchor_score", "constraint_coverage", "recovered_edges", "operator_count", "timeline_length"]
handoff_matrix = handoff_features[feature_cols].to_numpy(dtype=float)

print("Phase-lock gate:", PHASE_LOCK_GATE)
print("Handoff feature columns:", feature_cols)
handoff_features[["case_id"] + feature_cols]

## 12. Final notebook status

In [ ]:
status = {
    "repo": REPO_NAME,
    "notebook": NOTEBOOK_ID,
    "cases": len(CASES),
    "mean_temporal_decomposition_score": round(float(mean_score), 3),
    "min_temporal_decomposition_score": round(float(min_score), 3),
    "cases_with_ambiguity_flags": num_flags,
    "phase_lock_gate_next": PHASE_LOCK_GATE,
    "artifacts": [str(cases_path), str(edges_path), str(scores_path), str(summary_path), str(fig1_path), str(fig2_path), str(fig3_path)],
}
print(json.dumps(status, indent=2))